In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
DB = Path("music.db")
CSV = Path("data/CleanedData.csv")

In [3]:
# FIX #7: PRAGMA foreign_keys is per-connection, not stored in the file.
# This helper must be used for every connection.

def get_conn(path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON;")   # must be outside a transaction
    conn.execute("PRAGMA journal_mode = WAL;")
    conn.execute("PRAGMA busy_timeout = 5000;")
    return conn


df = pd.read_csv(CSV)
df = df.reset_index(drop=True)
df["track_key"] = df.index.astype(int)

In [4]:
# ------------------------------------------------------------------
# Split frames
# ------------------------------------------------------------------
metadata_cols = [
    "track_key", "track_id", "track_name", "artists",
    "album_name", "track_genre", "popularity", "duration_m", "explicit",
]
audio_cols = [
    "track_key", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature",
]

In [5]:
df_meta = df[metadata_cols].rename(
    columns={"track_id": "spotify_track_id", "track_genre": "primary_genre"}
)
df_audio = df[audio_cols].copy()

# Cast integer-like columns to Python int so SQLite stores INTEGER, not REAL
for col in ["popularity", "explicit"]:
    df_meta[col] = df_meta[col].astype(int)
for col in ["key", "mode", "time_signature"]:
    df_audio[col] = df_audio[col].astype(int)

In [6]:
# ------------------------------------------------------------------
# FIX #1 (part B): explode all_genres into a proper 1:N junction table
# ------------------------------------------------------------------
genres = (
    df[["track_key", "all_genres"]]
    .assign(track_genre=lambda d: d["all_genres"].str.split(";"))
    .explode("track_genre")
    .loc[lambda d: d["track_genre"].ne("")]
    [["track_key", "track_genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [7]:
# ------------------------------------------------------------------
# FIX #8: idempotent rebuild — drop child tables first, then parent.
# ------------------------------------------------------------------
conn = get_conn(DB)
try:
    with conn:
        conn.execute("DROP TABLE IF EXISTS track_genres;")
        conn.execute("DROP TABLE IF EXISTS audio_features;")
        conn.execute("DROP TABLE IF EXISTS tracks_metadata;")

        conn.execute("""
        CREATE TABLE tracks_metadata (
            track_key        INTEGER PRIMARY KEY,
            spotify_track_id TEXT UNIQUE NOT NULL,
            track_name       TEXT,
            artists          TEXT,
            album_name       TEXT,
            primary_genre    TEXT,
            popularity       INTEGER CHECK (popularity BETWEEN 0 AND 100),
            duration_m       REAL,
            explicit         INTEGER CHECK (explicit IN (0, 1))
        );
        """)

        conn.execute("""
        CREATE TABLE audio_features (
            track_key        INTEGER PRIMARY KEY,
            danceability     REAL,
            energy           REAL,
            key              INTEGER CHECK (key BETWEEN -1 AND 11),
            loudness         REAL,
            mode             INTEGER CHECK (mode IN (0, 1)),
            speechiness      REAL,
            acousticness     REAL,
            instrumentalness REAL,
            liveness         REAL,
            valence          REAL,
            tempo            REAL,
            time_signature   INTEGER CHECK (time_signature BETWEEN 3 AND 7),
            FOREIGN KEY (track_key) REFERENCES tracks_metadata(track_key)
                ON DELETE CASCADE
        );
        """)

        conn.execute("""
        CREATE TABLE track_genres (
            track_key   INTEGER NOT NULL,
            track_genre TEXT    NOT NULL,
            PRIMARY KEY (track_key, track_genre),
            FOREIGN KEY (track_key) REFERENCES tracks_metadata(track_key)
                ON DELETE CASCADE
        );
        """)

        # FIX #9: indexes for the obvious downstream filters
        conn.execute("CREATE INDEX idx_tracks_artists ON tracks_metadata(artists);")
        conn.execute("CREATE INDEX idx_track_genres_genre ON track_genres(track_genre);")
finally:
    conn.close()

In [8]:
# ------------------------------------------------------------------
# Load — parent first, then children (FK order matters)
# ------------------------------------------------------------------
conn = get_conn(DB)
try:
    with conn:
        df_meta.to_sql("tracks_metadata", conn, if_exists="append", index=False)
        df_audio.to_sql("audio_features", conn, if_exists="append", index=False)
        genres.to_sql("track_genres", conn, if_exists="append", index=False)
finally:
    conn.close()